# Symbolic Hamiltonian and diffusion with fpgen

This notebook constructs representative nonlinear open quantum systems and uses **fpgen** to derive their Wigner Kramers–Moyal series, truncated Fokker–Planck equation, effective Hamiltonian, and diffusion matrices.

For a mode vector $\boldsymbol{\alpha}$, the displayed equation is

$$\dot{\boldsymbol{\alpha}}=-i\,\mathbf H(\boldsymbol{\alpha})\boldsymbol{\alpha}+\boldsymbol{\xi},$$

with normal and anomalous correlations

$$\mathbf D=\langle\boldsymbol{\xi}\boldsymbol{\xi}^{\dagger}\rangle,\qquad \mathbf M=\langle\boldsymbol{\xi}\boldsymbol{\xi}^{T}\rangle.$$

All model parameters are declared real. For Wigner/P/Q, $D$ and $M$ are read directly from the corresponding blocks of the augmented FPE diffusion matrix, without reconstructing them through a noise factor $B$.

In [15]:
import sympy as sp
from IPython.display import Math, display
from fpgen import boson_modes, MasterEquation, LindbladChannel, derive_kramers_moyal

a, b, c = boson_modes("a", "b", "c")
omega_a, omega_b, omega_c = sp.symbols("omega_a omega_b omega_c", real=True)
g, g_ab, g_ac, k, chi = sp.symbols("g g_ab g_ac k chi", real=True)
gamma_a, gamma_b, gamma_c, Gamma = sp.symbols("gamma_a gamma_b gamma_c Gamma", real=True)

_DISPLAY_BASES = (r"\alpha", r"\beta", r"\gamma", r"\delta")

def _display_matrix(matrix, variables):
    n = len(variables) // 2
    replacements = {}
    for i in range(n):
        base = _DISPLAY_BASES[i]
        replacements[variables[i]] = sp.Symbol(base, complex=True)
        replacements[variables[n + i]] = sp.Symbol(base + r"^{*}", complex=True)
    return sp.Matrix(matrix).applyfunc(sp.simplify).xreplace(replacements)

def derive_and_display(master):
    km = derive_kramers_moyal(master, representation="wigner")
    fpe = km.truncate(2)
    langevin = fpe.to_langevin()

    H = langevin.physical_hamiltonian
    D = langevin.normal_correlation
    M = langevin.anomalous_correlation
    if H is None:
        raise ValueError("The physical Hamiltonian block is not closed for this model.")

    status = "exact" if fpe.is_exact else f"truncated from KM order {km.order}"
    print(f"Fokker–Planck status: {status}")
    display(Math(r"\mathbf H(\boldsymbol{\alpha})=" + sp.latex(_display_matrix(H, langevin.variables))))
    display(Math(r"\mathbf D(\boldsymbol{\alpha})=" + sp.latex(_display_matrix(D, langevin.variables))))
    display(Math(r"\mathbf M(\boldsymbol{\alpha})=" + sp.latex(_display_matrix(M, langevin.variables))))
    return km, fpe, langevin

## 1. Two-mode van der Pol oscillator

$$H=\omega_a a^\dagger a+\omega_b b^\dagger b+g(a^\dagger b+ab^\dagger),$$

with linear gain $\gamma_a\mathcal L[a^\dagger]$, nonlinear loss $\Gamma\mathcal L[a^2]$, and linear loss $\gamma_b\mathcal L[b]$.

In [16]:
master_vdp = MasterEquation(
    modes=[a, b],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b
        + g*(a.dag*b + a*b.dag)
    ),
    channels=[
        LindbladChannel(a.dag, gamma_a),
        LindbladChannel(a**2, Gamma),
        LindbladChannel(b, gamma_b),
    ],
)
km_vdp, fpe_vdp, langevin_vdp = derive_and_display(master_vdp)

Fokker–Planck status: truncated from KM order 3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Two-mode pair hopping

$$H=\omega_a a^\dagger a+\omega_b b^\dagger b+g(a^\dagger b+ab^\dagger)+k(a^{\dagger 2}b^2+b^{\dagger 2}a^2),$$

with gain on $a$ and loss on $b$.

In [17]:
master_pair = MasterEquation(
    modes=[a, b],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b
        + g*(a.dag*b + a*b.dag)
        + k*(a.dag**2*b**2 + b.dag**2*a**2)
    ),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_pair, fpe_pair, langevin_pair = derive_and_display(master_pair)

Fokker–Planck status: truncated from KM order 3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Two-mode Kerr system

$$H=\omega_a a^\dagger a+\omega_b b^\dagger b+g(a^\dagger b+ab^\dagger)+\chi a^{\dagger 2}a^2,$$

with gain on $a$ and loss on $b$.

In [21]:
master_kerr2 = MasterEquation(
    modes=[a, b],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b
        + g*(a.dag*b + a*b.dag)
        + chi*a.dag**2*a**2
    ),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_kerr2, fpe_kerr2, langevin_kerr2 = derive_and_display(master_kerr2)

Fokker–Planck status: truncated from KM order 3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Three-mode Kerr system

Modes $a$ and $b$ are lossy, mode $c$ has gain, and the Kerr nonlinearity acts on mode $a$.

In [19]:
master_kerr3 = MasterEquation(
    modes=[a, b, c],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b + omega_c*c.dag*c
        + g_ab*(a.dag*b + a*b.dag)
        + g_ac*(a.dag*c + a*c.dag)
        + chi*a.dag**2*a**2
    ),
    channels=[
        LindbladChannel(a, gamma_a),
        LindbladChannel(b, gamma_b),
        LindbladChannel(c.dag, gamma_c),
    ],
)
km_kerr3, fpe_kerr3, langevin_kerr3 = derive_and_display(master_kerr3)

Fokker–Planck status: truncated from KM order 3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 5. Two-mode cross-Kerr system

$$H=\omega_a a^\dagger a+\omega_b b^\dagger b+g(a^\dagger b+ab^\dagger)+\chi a^\dagger a b^\dagger b,$$

with gain on $a$ and loss on $b$.

In [20]:
master_cross = MasterEquation(
    modes=[a, b],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b
        + g*(a.dag*b + a*b.dag)
        + chi*a.dag*a*b.dag*b
    ),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_cross, fpe_cross, langevin_cross = derive_and_display(master_cross)

Fokker–Planck status: truncated from KM order 3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>